# NB05 — BanglaBERT Baseline (leakage-controlled)

Full fine-tuning of `csebuetnlp/banglabert` on all four task variants. Clean reference point for v2. Runs on **RunPod (GPU)**.

Defense-in-depth: every split is re-checked for train/val/test leakage at load time — the notebook hard-fails if any overlap exists.

In [1]:
# Ensure dependencies (fast no-op if already installed on the pod)
import importlib, subprocess, sys
def ensure(pip_name, import_name=None):
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        print(f"installing {pip_name} ...")
        subprocess.run([sys.executable,"-m","pip","install","-q",pip_name], check=False)
for pkg,imp in [("transformers","transformers"),("datasets","datasets"),
                ("accelerate","accelerate"),("scikit-learn","sklearn"),
                ("pandas","pandas"),("numpy","numpy")]:
    ensure(pkg,imp)
print("deps ok")


deps ok


In [2]:
import os, json, time, random, shutil, warnings, unicodedata, re, gc
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, classification_report)
warnings.filterwarnings("ignore")
import transformers
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


torch: 2.8.0+cu128 | transformers: 4.40.0
device: cuda
GPU: NVIDIA RTX A5000 25.4 GB


In [3]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("seed set:", SEED)


seed set: 42


In [4]:
def find_repo_root():
    cands = [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents)
    for c in cands:
        if (c / "01_data").exists():
            return c.resolve()
    raise RuntimeError("Could not locate repo root (no 01_data dir found). Run from inside the repo.")

ROOT       = find_repo_root()
SPLITS     = ROOT / "01_data" / "interim" / "splits"
CKPT       = ROOT / "03_checkpoints"
TABLES     = ROOT / "04_outputs" / "tables"
PRED       = ROOT / "04_outputs" / "predictions"
LOGS       = ROOT / "04_outputs" / "run_logs"
REPORTS    = ROOT / "04_outputs" / "reports"
for p in [CKPT, TABLES, PRED, LOGS, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)
print("ROOT  :", ROOT)
print("SPLITS:", SPLITS, "| exists:", SPLITS.exists())
assert SPLITS.exists(), f"Splits folder missing: {SPLITS}. Run NB01 first."


ROOT  : /workspace/Sarcasm_detection
SPLITS: /workspace/Sarcasm_detection/01_data/interim/splits | exists: True


In [5]:
# ---- normalized key + leakage guard (defense-in-depth; NB01 already enforces this) ----
_ZW = dict.fromkeys(map(ord, ["\u200b","\u200c","\u200d","\ufeff"]), None)
def norm_key(s):
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFC", s).translate(_ZW)
    s = re.sub(r"\s+", " ", s).strip()
    return s.casefold()

def assert_no_internal_leakage(tr, va, te, text_col="text"):
    s_tr = {norm_key(x) for x in tr[text_col]}
    s_va = {norm_key(x) for x in va[text_col]}
    s_te = {norm_key(x) for x in te[text_col]}
    n1, n2, n3 = len(s_tr & s_va), len(s_tr & s_te), len(s_va & s_te)
    assert n1 == 0, f"LEAK train/val overlap = {n1}"
    assert n2 == 0, f"LEAK train/test overlap = {n2}"
    assert n3 == 0, f"LEAK val/test overlap = {n3}"
    return {"train_val": n1, "train_test": n2, "val_test": n3}

def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def eval_from_logits(labels, logits, num_labels):
    labels = np.asarray(labels)
    preds = np.asarray(logits).argmax(-1)
    m = {"accuracy": float(accuracy_score(labels, preds)),
         "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
         "weighted_f1": float(f1_score(labels, preds, average="weighted", zero_division=0))}
    if num_labels == 2:
        m["binary_f1"]  = float(f1_score(labels, preds, average="binary", pos_label=1, zero_division=0))
        m["precision"]  = float(precision_score(labels, preds, average="binary", pos_label=1, zero_division=0))
        m["recall"]     = float(recall_score(labels, preds, average="binary", pos_label=1, zero_division=0))
    return m, preds

def make_compute_metrics(num_labels):
    def _cm(eval_pred):
        logits, labels = eval_pred
        preds = np.asarray(logits).argmax(-1)
        return {"accuracy": accuracy_score(labels, preds),
                "macro_f1": f1_score(labels, preds, average="macro", zero_division=0)}
    return _cm

def save_predictions(path, texts, labels, logits, num_labels, extra=None):
    logits = np.asarray(logits); probs = softmax_np(logits); preds = logits.argmax(-1)
    d = {"text": [str(t) for t in texts], "gold_label": np.asarray(labels),
         "pred_label": preds, "correct": (np.asarray(labels) == preds).astype(int)}
    for k in range(num_labels): d[f"logit_{k}"] = logits[:, k]
    for k in range(num_labels): d[f"prob_{k}"]  = probs[:, k]
    df = pd.DataFrame(d)
    if extra:
        for k, v in extra.items(): df[k] = v
    df.to_csv(path, index=False)

class TorchTextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.enc = tokenizer(list(texts), truncation=True, padding="max_length",
                             max_length=max_length, return_tensors="pt")
        self.labels = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

def build_args(output_dir, epochs, batch, lr, wd, warmup, fp16, metric="macro_f1"):
    common = dict(output_dir=str(output_dir), num_train_epochs=epochs,
                  per_device_train_batch_size=batch, per_device_eval_batch_size=batch*2,
                  learning_rate=lr, weight_decay=wd, warmup_ratio=warmup,
                  save_strategy="epoch", logging_strategy="epoch", save_total_limit=1,
                  load_best_model_at_end=True, metric_for_best_model=metric,
                  greater_is_better=True, report_to="none", seed=SEED, data_seed=SEED,
                  fp16=fp16, bf16=False)
    try:
        return TrainingArguments(evaluation_strategy="epoch", **common)
    except TypeError:
        return TrainingArguments(eval_strategy="epoch", **common)


In [6]:
TASKS = {
    "ben_sarc_binary":     {"label_col": "label_binary",  "num_labels": 2},
    "banglasarc_binary":   {"label_col": "label_binary",  "num_labels": 2},
    "banglasarc3_binary":  {"label_col": "label_binary",  "num_labels": 2},
    "banglasarc3_ternary": {"label_col": "label_ternary", "num_labels": 3},
}

def load_task(task):
    cfg = TASKS[task]; lc = cfg["label_col"]
    def rd(split):
        df = pd.read_csv(SPLITS / f"{task}_{split}.csv")
        assert "text" in df.columns,  f"{task}_{split}: missing 'text'"
        assert lc in df.columns,      f"{task}_{split}: missing '{lc}'"
        df = df[["text", lc]].dropna(subset=["text"]).copy()
        df["text"] = df["text"].astype(str)
        df[lc] = df[lc].astype(int)
        return df
    tr, va, te = rd("train"), rd("val"), rd("test")
    leak = assert_no_internal_leakage(tr, va, te)   # hard fail on leakage
    return tr, va, te, lc, cfg["num_labels"], leak


In [7]:
# ---- CONFIG ----
MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS     = 4
LR         = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
PATIENCE   = 2
USE_FP16   = torch.cuda.is_available()
SAVE_BEST_WEIGHTS = True
print("model:", MODEL_NAME, "| epochs:", EPOCHS, "| batch:", BATCH_SIZE, "| fp16:", USE_FP16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


model: csebuetnlp/banglabert | epochs: 4 | batch: 32 | fp16: True


tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [8]:
def run_task(task):
    print("\n" + "="*78 + f"\nTASK: {task}\n" + "="*78)
    tr, va, te, lc, num_labels, leak = load_task(task)
    print(f"sizes -> train {len(tr)} | val {len(va)} | test {len(te)} | leak {leak}")

    train_ds = TorchTextDataset(tr["text"], tr[lc], tokenizer, MAX_LENGTH)
    val_ds   = TorchTextDataset(va["text"], va[lc], tokenizer, MAX_LENGTH)
    test_ds  = TorchTextDataset(te["text"], te[lc], tokenizer, MAX_LENGTH)

    out_dir = CKPT / f"05_baseline_banglabert_{task}"
    if out_dir.exists(): shutil.rmtree(out_dir, ignore_errors=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    args  = build_args(out_dir, EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY, WARMUP_RATIO, USE_FP16)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      compute_metrics=make_compute_metrics(num_labels),
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    t0 = time.time()
    train_out = trainer.train()
    secs = round(time.time() - t0, 1)

    val_logits  = trainer.predict(val_ds).predictions
    test_logits = trainer.predict(test_ds).predictions
    val_m,  _ = eval_from_logits(va[lc].values, val_logits,  num_labels)
    test_m, test_preds = eval_from_logits(te[lc].values, test_logits, num_labels)
    print("VAL :", {k: round(v,4) for k,v in val_m.items()})
    print("TEST:", {k: round(v,4) for k,v in test_m.items()})

    save_predictions(PRED / f"05_baseline_banglabert_{task}_val_predictions.csv",
                     va["text"].values, va[lc].values, val_logits, num_labels,
                     extra={"task": task, "split": "val", "model": MODEL_NAME})
    save_predictions(PRED / f"05_baseline_banglabert_{task}_test_predictions.csv",
                     te["text"].values, te[lc].values, test_logits, num_labels,
                     extra={"task": task, "split": "test", "model": MODEL_NAME})

    cm = confusion_matrix(te[lc].values, test_preds).tolist()
    json.dump({"test": cm}, open(TABLES / f"05_baseline_banglabert_{task}_confusion.json","w"), indent=2)

    if SAVE_BEST_WEIGHTS:
        best_dir = CKPT / f"05_baseline_banglabert_{task}" / "best_model"
        best_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(best_dir)); tokenizer.save_pretrained(str(best_dir))

    row = {"task": task, "model": "banglabert", "model_name": MODEL_NAME, "num_labels": num_labels,
           "n_train": len(tr), "n_val": len(va), "n_test": len(te),
           "epochs": EPOCHS, "batch": BATCH_SIZE, "lr": LR, "max_length": MAX_LENGTH,
           "time_seconds": secs, "leak_train_val": leak["train_val"],
           "leak_train_test": leak["train_test"], "leak_val_test": leak["val_test"]}
    for k,v in val_m.items():  row[f"val_{k}"]  = v
    for k,v in test_m.items(): row[f"test_{k}"] = v

    del model, trainer; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    shutil.rmtree(out_dir, ignore_errors=True)   # drop intermediate epoch ckpts; keep best_model
    return row


In [9]:
SUMMARY = TABLES / "05_baseline_banglabert_summary.csv"
done = set()
rows = []
if SUMMARY.exists():
    prev = pd.read_csv(SUMMARY); rows = prev.to_dict("records"); done = set(prev["task"])
    print("resuming, done:", done)

for task in TASKS:
    if task in done:
        print("skip (done):", task); continue
    rows.append(run_task(task))
    pd.DataFrame(rows).to_csv(SUMMARY, index=False)
    print("saved ->", SUMMARY.name)

summary_df = pd.DataFrame(rows)
print("\nALL DONE\n")
cols = ["task","test_accuracy","test_macro_f1","test_weighted_f1","time_seconds"]
print(summary_df[cols].to_string(index=False))



TASK: ben_sarc_binary
sizes -> train 20498 | val 2562 | test 2563 | leak {'train_val': 0, 'train_test': 0, 'val_test': 0}


pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.537600,0.442953,0.791569,0.791386
2,0.378100,0.449474,0.799375,0.798991
3,0.257800,0.584068,0.777127,0.775036
4,0.172400,0.646683,0.795472,0.794860


VAL : {'accuracy': 0.7994, 'macro_f1': 0.799, 'weighted_f1': 0.799, 'binary_f1': 0.7902, 'precision': 0.8281, 'recall': 0.7557}
TEST: {'accuracy': 0.803, 'macro_f1': 0.8025, 'weighted_f1': 0.8025, 'binary_f1': 0.7928, 'precision': 0.8364, 'recall': 0.7535}
saved -> 05_baseline_banglabert_summary.csv

TASK: banglasarc_binary
sizes -> train 3708 | val 463 | test 464 | leak {'train_val': 0, 'train_test': 0, 'val_test': 0}


...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.613400,0.521184,0.762326,0.762317
2,0.472600,0.518058,0.764855,0.764791
3,0.350000,0.591132,0.758534,0.758272
4,0.262000,0.631504,0.757269,0.756481


VAL : {'accuracy': 0.7649, 'macro_f1': 0.7648, 'weighted_f1': 0.7648, 'binary_f1': 0.7687, 'precision': 0.7555, 'recall': 0.7823}
TEST: {'accuracy': 0.7535, 'macro_f1': 0.7534, 'weighted_f1': 0.7534, 'binary_f1': 0.7566, 'precision': 0.7463, 'recall': 0.7671}
saved -> 05_baseline_banglabert_summary.csv

TASK: banglasarc3_ternary
sizes -> train 9528 | val 1191 | test 1192 | leak {'train_val': 0, 'train_test': 0, 'val_test': 0}


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.952100,0.864545,0.599496,0.580190
2,0.752800,0.806735,0.628044,0.624494
3,0.603300,0.879631,0.635600,0.638324
4,0.489600,0.903911,0.634761,0.635867


VAL : {'accuracy': 0.6356, 'macro_f1': 0.6383, 'weighted_f1': 0.6384}
TEST: {'accuracy': 0.6317, 'macro_f1': 0.6331, 'weighted_f1': 0.633}
saved -> 05_baseline_banglabert_summary.csv

ALL DONE

               task  test_accuracy  test_macro_f1  test_weighted_f1  time_seconds
    ben_sarc_binary       0.802965       0.802488          0.802484         249.7
  banglasarc_binary       0.978448       0.976550          0.978448          61.4
 banglasarc3_binary       0.753477       0.753437          0.753433          92.4
banglasarc3_ternary       0.631711       0.633064          0.633034         124.2


In [10]:
# leakage receipt (all zeros expected)
print("Leakage check (must be all 0):")
print(summary_df[["task","leak_train_val","leak_train_test","leak_val_test"]].to_string(index=False))


Leakage check (must be all 0):
               task  leak_train_val  leak_train_test  leak_val_test
    ben_sarc_binary               0                0              0
  banglasarc_binary               0                0              0
 banglasarc3_binary               0                0              0
banglasarc3_ternary               0                0              0
